In [3]:
from dotenv import load_dotenv
load_dotenv()
from pprint import pprint

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool

In [4]:
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [ ]:
# create subagents

MODEL_NAME = "openai:gpt-5-nano"

subagent1 = create_agent(
    model=MODEL_NAME,
    tools=[square_root],
)

subagent2 = create_agent(
    model=MODEL_NAME,
    tools=[square],
)

In [7]:
# calling subagents

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent1.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square root of {x}")]}
    )
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 1 in order to calculate the square of a number"""
    response = subagent2.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square of {x}")]}
    )
    return response["messages"][-1].content

In [8]:
# Creating the main agent

SYSTEM_PROMPT = "You are a helpful assistant who can call subagents to calculate the square root of square of a number"

main_agent = create_agent(
    model=MODEL_NAME,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt=SYSTEM_PROMPT,
)

In [9]:
# Test the agents

questions = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(questions)]})
pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='3f8ee014-131f-4df7-b837-768df233c1a0'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2907, 'prompt_tokens': 202, 'total_tokens': 3109, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2880, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DFWDY3DhOO1vQFROsi8T4dBixfsS0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cb6aa-39f4-7652-b67b-6b613793e9cb-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'call_KSxaZdoiGZeCoa7YHFIOWFw0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 202,